# Phase 5 — End-to-End Project Test

## How to Run the Full Project

This notebook verifies that every part of the pipeline is working before you open the dashboard.

### Complete startup sequence

**Step 1 — Start infrastructure**
```bash
cd fifa-pipeline/
docker-compose up --build
# Wait ~60 seconds for all services to become healthy
```

**Step 2 — Run the pipeline (first time)**
```bash
# Option A: Run notebooks one by one (development mode)
jupyter notebook notebooks/01_ingestion.ipynb
jupyter notebook notebooks/02_spark_processing.ipynb
jupyter notebook notebooks/03_feature_engineering.ipynb
jupyter notebook notebooks/02b_dbt_runner.ipynb
jupyter notebook notebooks/04_ml_model.ipynb

# Option B: Trigger the Airflow DAG (production mode)
# 1. Open http://localhost:8080 → login with admin/admin
# 2. Find 'fifa_wc2026_pipeline' → toggle ON → click 'Trigger DAG'
# 3. Watch all 6 tasks turn green
```

**Step 3 — Open the dashboard**
```bash
# If running locally (not via Docker):
streamlit run dashboard/app.py

# Via Docker (already running after docker-compose up):
# Open http://localhost:8501
```

**Step 4 — Run this notebook to verify everything**
```bash
jupyter notebook notebooks/05_dashboard_test.ipynb
```

---

## Architecture Recap

```
data/raw/*.csv
    ↓ [Phase 1] pandas + SQLAlchemy
PostgreSQL: raw_results, raw_goalscorers, raw_shootouts
    ↓ [Phase 2] PySpark window functions
PostgreSQL: match_features  +  data/processed/*.parquet
    ↓ [Phase 3] dbt models
PostgreSQL: stg_* views, mart_team_form, mart_head_to_head, mart_predictions_input
    ↓ [Phase 4] XGBoost
PostgreSQL: wc2026_predictions  +  models/xgb_fifa_model.pkl
    ↓ [Phase 5] Streamlit + Airflow
Dashboard: http://localhost:8501
Orchestration: http://localhost:8080
```

In [ ]:
# ============================================================
# Cell 2 — Test all PostgreSQL tables exist and have data
# ============================================================
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, inspect

load_dotenv(dotenv_path=Path("..") / ".env")
DB_URL = os.getenv("DATABASE_URL")

print("Connecting to PostgreSQL...")
engine = create_engine(DB_URL)

with engine.connect() as conn:
    pg_version = conn.execute(text("SELECT version();")).fetchone()[0]
print(f"Connected! PostgreSQL version: {pg_version[:50]}...\n")

# ---- Expected tables ----
# These should all exist and have data after running the full pipeline.
expected_tables = {
    # Phase 1 — Raw ingestion
    "raw_results":         {"min_rows": 40000, "key_col": "home_team"},
    "raw_goalscorers":     {"min_rows": 10000, "key_col": "scorer"},
    "raw_shootouts":       {"min_rows": 100,   "key_col": "winner"},
    "raw_former_names":    {"min_rows": 10,    "key_col": "current"},
    # Phase 2 — PySpark feature table
    "match_features":      {"min_rows": 20000, "key_col": "result"},
    # Phase 3 — dbt seed
    "wc2026_fixtures":     {"min_rows": 48,    "key_col": "fixture_id"},
    # Phase 4 — ML predictions
    "wc2026_predictions":  {"min_rows": 48,    "key_col": "predicted_result"},
}

# dbt creates tables in separate schemas — check both
dbt_mart_tables = {
    "mart_team_form":         {"schema": "marts",   "min_rows": 50},
    "mart_head_to_head":      {"schema": "marts",   "min_rows": 100},
    "mart_predictions_input": {"schema": "marts",   "min_rows": 48},
}

print(f"{'Table':<35} {'Rows':>8} {'Min Expected':>14} {'Status':>8}")
print("-" * 70)

all_pass = True

# Check public schema tables
for table, config in expected_tables.items():
    try:
        with engine.connect() as conn:
            count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).fetchone()[0]
        min_r  = config["min_rows"]
        status = "PASS" if count >= min_r else "LOW"
        if count < min_r:
            all_pass = False
    except Exception as e:
        count  = 0
        status = "MISSING"
        all_pass = False
        min_r  = config["min_rows"]
    
    icon = "OK" if status == "PASS" else ("!!" if status == "MISSING" else "??")
    print(f"  [{icon}] {table:<31} {count:>8,} {min_r:>14,} {status:>8}")

# Check dbt mart tables
for table, config in dbt_mart_tables.items():
    schema = config["schema"]
    min_r  = config["min_rows"]
    for schema_try in [schema, "public"]:
        try:
            with engine.connect() as conn:
                count = conn.execute(
                    text(f"SELECT COUNT(*) FROM {schema_try}.{table}")
                ).fetchone()[0]
            status = "PASS" if count >= min_r else "LOW"
            break
        except Exception:
            count  = 0
            status = "MISSING"
    else:
        all_pass = False

    icon = "OK" if status == "PASS" else ("!!" if status == "MISSING" else "??")
    display_name = f"{schema_try}.{table}"
    print(f"  [{icon}] {display_name:<31} {count:>8,} {min_r:>14,} {status:>8}")

print("-" * 70)
print(f"\n  Overall: {'ALL TABLES OK' if all_pass else 'SOME TABLES MISSING OR EMPTY'}")

if not all_pass:
    print("""
  To fix missing tables:
    Phase 1 tables (raw_*):         Run notebook 01_ingestion.ipynb
    match_features:                 Run notebooks 02 + 03
    wc2026_fixtures:                Run: cd dbt && dbt seed --profiles-dir .
    mart_* tables:                  Run: cd dbt && dbt run  --profiles-dir .
    wc2026_predictions:             Run notebook 04_ml_model.ipynb
""")

## Testing the Streamlit Dashboard

We send an HTTP GET request to `http://localhost:8501` and check for a 200 response.  
This confirms:
1. The Streamlit process is running
2. It's accessible on the expected port
3. It didn't crash on startup (a 200 implies the Python code at least loaded)

**If this fails:** make sure you started Streamlit:
- Via Docker: `docker-compose up streamlit`
- Locally: `streamlit run dashboard/app.py`

In [ ]:
# ============================================================
# Cell 3 — Test Streamlit app is reachable
# ============================================================
import requests
import time

STREAMLIT_URL = "http://localhost:8501"

print(f"Testing Streamlit at {STREAMLIT_URL} ...")

max_retries = 3
for attempt in range(1, max_retries + 1):
    try:
        resp = requests.get(STREAMLIT_URL, timeout=10)
        if resp.status_code == 200:
            print(f"  [OK] Streamlit responded with HTTP {resp.status_code}")
            print(f"  Content-Type: {resp.headers.get('Content-Type', 'N/A')}")
            print(f"  Response size: {len(resp.content):,} bytes")
            print(f"\n  Dashboard is running! Open: {STREAMLIT_URL}")
            break
        else:
            print(f"  [!!] Attempt {attempt}: HTTP {resp.status_code}")
    except requests.exceptions.ConnectionError:
        print(f"  [!!] Attempt {attempt}: Connection refused.")
        if attempt < max_retries:
            print(f"      Retrying in 5 seconds...")
            time.sleep(5)
    except requests.exceptions.Timeout:
        print(f"  [!!] Attempt {attempt}: Timeout after 10s.")
else:
    print(f"""
  Dashboard not reachable. To start it:
  
  Option A (local):
    cd fifa-pipeline/
    streamlit run dashboard/app.py
  
  Option B (Docker):
    docker-compose up streamlit -d
    # Wait ~30 seconds for packages to install, then retry.
""")

# ---- Also test Airflow ----
AIRFLOW_URL = "http://localhost:8080/health"
print(f"\nTesting Airflow at {AIRFLOW_URL} ...")
try:
    resp_af = requests.get(AIRFLOW_URL, timeout=10)
    if resp_af.status_code == 200:
        health = resp_af.json()
        print(f"  [OK] Airflow health: {health}")
    else:
        print(f"  [!!] Airflow returned HTTP {resp_af.status_code}")
except Exception as e:
    print(f"  [!!] Airflow not reachable: {e}")
    print("  Start with: docker-compose up airflow-webserver airflow-scheduler -d")

## Full Project Summary

This final cell prints a complete end-to-end summary of everything the pipeline built:
- All database table row counts
- Model accuracy and feature count
- Number of predictions made
- Most confident predictions
- Group-by-group predicted winners

In [ ]:
# ============================================================
# Cell 4 — Full project summary
# ============================================================
import json
import joblib
from pathlib import Path
from sqlalchemy import create_engine, text

engine = create_engine(DB_URL)

print("=" * 65)
print("  FIFA WC 2026 PREDICTION PIPELINE — PROJECT SUMMARY")
print("=" * 65)

# ---- Database tables ----
print("\n DATABASE TABLES")
print("-" * 50)

tables_to_check = [
    ("public",  "raw_results"),
    ("public",  "raw_goalscorers"),
    ("public",  "raw_shootouts"),
    ("public",  "match_features"),
    ("public",  "wc2026_fixtures"),
    ("public",  "wc2026_predictions"),
    ("marts",   "mart_team_form"),
    ("marts",   "mart_head_to_head"),
    ("marts",   "mart_predictions_input"),
]

total_rows = 0
for schema, table in tables_to_check:
    for s in [schema, "public"]:
        try:
            with engine.connect() as conn:
                count = conn.execute(
                    text(f"SELECT COUNT(*) FROM {s}.{table}")
                ).fetchone()[0]
            total_rows += count
            print(f"  {s+'.'+table:<40} {count:>10,} rows")
            break
        except Exception:
            continue
    else:
        print(f"  {schema+'.'+table:<40} {'NOT FOUND':>10}")

print(f"\n  Total rows across all tables: {total_rows:,}")

# ---- Model info ----
print("\n\n ML MODEL")
print("-" * 50)
meta_path = Path("..") / "models" / "model_metadata.json"
if meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)
    print(f"  Algorithm       : XGBoost multi-class classifier")
    print(f"  Classes         : W (Win), D (Draw), L (Loss)")
    print(f"  Features        : {len(meta['feature_cols'])} columns")
    print(f"  Training samples: {meta['n_training_samples']:,}")
    print(f"  Test accuracy   : {meta['test_accuracy']*100:.2f}%")
    print(f"  Best iteration  : {meta['best_iteration']} trees")
    print(f"  Trained on      : {meta['training_date'][:10]}")
    print(f"\n  Feature columns:")
    for i, feat in enumerate(meta['feature_cols'], 1):
        print(f"    {i:2}. {feat}")
else:
    print("  Model metadata not found. Run notebook 04_ml_model.ipynb.")

# ---- Predictions summary ----
print("\n\n PREDICTIONS")
print("-" * 50)
try:
    df_pred = pd.read_sql("SELECT * FROM wc2026_predictions", engine)
    print(f"  Total predictions: {len(df_pred)}")
    print(f"  Avg confidence  : {df_pred['confidence'].mean()*100:.1f}%")
    print(f"  Predicted at    : {df_pred['predicted_at'].max()[:19]}")
    
    dist = df_pred["predicted_result"].value_counts()
    print(f"\n  Result distribution across 48 fixtures:")
    for r, cnt in dist.items():
        bar = "█" * cnt
        print(f"    {r}: {bar} ({cnt})")
    
    print("\n  Top 5 most confident predictions:")
    top5 = df_pred.sort_values("confidence", ascending=False).head(5)
    for _, r in top5.iterrows():
        label = {"W": "WIN", "D": "DRAW", "L": "LOSS"}.get(r["predicted_result"], "?")
        print(f"    {r['home_team']:<18} vs {r['away_team']:<18} → {label}  ({r['confidence']*100:.1f}%)")
    
    print("\n  Group-by-group prediction summary:")
    for group in sorted(df_pred["group_name"].unique()):
        g = df_pred[df_pred["group_name"] == group]
        results_str = " | ".join(
            f"{r['home_team'][:3]}-{r['away_team'][:3]}: {r['predicted_result']}"
            for _, r in g.sort_values("match_date").iterrows()
        )
        print(f"    Group {group}: {results_str}")

except Exception as e:
    print(f"  Could not load predictions: {e}")

# ---- File artifacts ----
print("\n\n FILE ARTIFACTS")
print("-" * 50)
artifacts = [
    ("models/xgb_fifa_model.pkl",  "Trained XGBoost classifier"),
    ("models/scaler.pkl",          "StandardScaler (fitted on train set)"),
    ("models/model_metadata.json", "Feature names, accuracy, label map"),
    ("data/processed/match_features.parquet", "Cleaned + featured match history"),
    ("data/processed/matches_clean.parquet",  "Cleaned raw results"),
]
for rel_path, description in artifacts:
    full_path = Path("..") / rel_path
    exists = "EXISTS" if full_path.exists() else "MISSING"
    size   = f"{full_path.stat().st_size / 1024:.0f} KB" if full_path.exists() else "-"
    print(f"  [{exists:7}] {rel_path:<45} {size:>8}   {description}")

# ---- Access URLs ----
print("\n\n ACCESS URLS")
print("-" * 50)
print("  Streamlit Dashboard  : http://localhost:8501")
print("  Airflow UI           : http://localhost:8080  (admin/admin)")
print("  PostgreSQL           : localhost:5432  (see .env for credentials)")
print("  dbt Docs             : run 'dbt docs serve --profiles-dir . --port 8081'")
print()
print("=" * 65)
print("  Pipeline built with:")
print("  PySpark 3.5 | dbt-postgres 1.8 | XGBoost 2.0 |")
print("  Airflow 2.9 | Streamlit 1.35  | PostgreSQL 15 | Docker")
print("=" * 65)

engine.dispose()